In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def select_ar_lag(series, max_p, criterion):
    df = pd.DataFrame({"y": series})

    # Create all lags up to max_p
    for i in range(1, max_p + 1):
        df[f"lag_{i}"] = df["y"].shift(i)
    
    df = df.dropna()  # ONE common sample
    
    y = df["y"]
    
    best_p = None
    best_ic = np.inf
    
    for p in range(1, max_p + 1):
        X = df[[f"lag_{i}" for i in range(1, p + 1)]]
        X = sm.add_constant(X)
        
        model = sm.OLS(y, X).fit()
        
        ic = model.bic if criterion == "bic" else model.aic
        
        if ic < best_ic:
            best_ic = ic
            best_p = p
    
    return best_p

In [4]:
df = pd.read_csv("../data/fred_md.csv")
df["sasdate"] = pd.to_datetime(df["sasdate"])
df = df[(df["sasdate"].dt.year < 2020) & (df["sasdate"].dt.year >= 1959)]
bic_p = {}
aic_p = {}

for col in df.columns[1:]:  # Skip date column
    best_p = select_ar_lag(df[col], max_p=12, criterion="bic")
    bic_p[col] = best_p

In [5]:
df_qd = pd.read_csv("../data/fred_qd_X.csv")
df_qd["sasdate"] = pd.to_datetime(df_qd["sasdate"])
df_qd = df_qd[(df_qd["sasdate"].dt.year < 2020) & (df_qd["sasdate"].dt.year >= 1959)]

for col in df_qd.columns[1:]:  # Skip date column
    best_p = select_ar_lag(df_qd[col], max_p=5, criterion="bic")
    bic_p[col] = best_p

c:\Users\Aninda\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
c:\Users\Aninda\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
c:\Users\Aninda\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
c:\Users\Aninda\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
c:\Users\Aninda\AppData\Local\Programs\Python\Python314\Lib\site

In [11]:
bic_p

{'RPI_t_md': 1,
 'W875RX1_t_md': 1,
 'DPCERA3M086SBEA_t_md': 1,
 'CMRMTSPLx_t_md': 3,
 'RETAILx_t_md': 2,
 'INDPRO_t_md': 3,
 'IPFPNSS_t_md': 4,
 'IPFINAL_t_md': 4,
 'IPCONGD_t_md': 1,
 'IPDCONGD_t_md': 1,
 'IPNCONGD_t_md': 1,
 'IPBUSEQ_t_md': 4,
 'IPMAT_t_md': 3,
 'IPDMAT_t_md': 1,
 'IPNMAT_t_md': 3,
 'IPMANSICS_t_md': 3,
 'IPB51222S_t_md': 5,
 'IPFUELS_t_md': 2,
 'CUMFNS_t_md': 3,
 'HWI_t_md': 6,
 'HWIURATIO_t_md': 4,
 'CLF16OV_t_md': 1,
 'CE16OV_t_md': 4,
 'UNRATE_t_md': 4,
 'UEMPMEAN_t_md': 6,
 'UEMPLT5_t_md': 2,
 'UEMP5TO14_t_md': 3,
 'UEMP15OV_t_md': 4,
 'UEMP15T26_t_md': 4,
 'UEMP27OV_t_md': 7,
 'CLAIMSx_t_md': 1,
 'PAYEMS_t_md': 4,
 'USGOOD_t_md': 3,
 'CES1021000001_t_md': 1,
 'USCONS_t_md': 5,
 'MANEMP_t_md': 3,
 'DMANEMP_t_md': 3,
 'NDMANEMP_t_md': 3,
 'SRVPRD_t_md': 4,
 'USTPU_t_md': 4,
 'USWTRADE_t_md': 4,
 'USTRADE_t_md': 12,
 'USFIRE_t_md': 3,
 'USGOVT_t_md': 9,
 'CES0600000007_t_md': 3,
 'AWOTMAN_t_md': 1,
 'AWHMAN_t_md': 3,
 'HOUST_t_md': 3,
 'HOUSTNE_t_md': 5,
 'HOUSTM

In [10]:
df_p = pd.DataFrame(list(bic_p.items()), columns=["variable", "lag"])
df_p.to_csv("bic_lags.csv", index=False)